# Kaggle - Vietnamese Medical NER: Train + Inference

1 notebook duy nhat: fine-tune LoRA (BUOC 1) + chay inference + tao submit (BUOC 2).

**Thu tu thuc hien tren Kaggle:**
1. Cell 1-2: Cai dat + kiem tra GPU
2. Cell 3: Clone repo GitHub
3. Cell 4: Cau hinh (MODEL + INFERENCE_MODE)
4. Cell 5: Train LoRA (neu TRAIN = True)
5. Cell 6: Kiem tra adapter
6. Cell 7: Load model + adapter
7. Cell 8: Chay inference tren test set
8. Cell 9: Tao file submit .zip

## Cell 1 - Cai dat thu vien

In [ ]:
!pip install -q -U transformers accelerate peft trl bitsandbytes datasets
!pip install -q editdistance

## Cell 2 - Kiem tra GPU

In [ ]:
import torch, os
print(f"PyTorch: {torch.__version__}")
print(f"CUDA: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    gb = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f"VRAM: {gb:.1f} GB")

## Cell 3 - Lay code (3 cach)

Chon 1 trong 3 cach. KHUYEN NGHI cach 1 (Kaggle Dataset) hoac cach 2 (upload zip).

In [ ]:
import os, sys, shutil, zipfile

WORK_DIR = "/kaggle/working"
REPO_NAME = "Viettel-AI-Race"
REPO_DIR = f"{WORK_DIR}/{REPO_NAME}"

# ===== CHON CACH LAY CODE ============================
# METHOD = 0 -> git clone (can public repo + internet ON)
# METHOD = 1 -> Kaggle Dataset (Add data -> vtai-code)
# METHOD = 2 -> Upload zip len Kaggle Datasets roi extract
METHOD = 0
# =====================================================

def via_git():
    GITHUB_USER = "bphong-cyrus"
    url = f"https://github.com/{GITHUB_USER}/{REPO_NAME}.git"
    print(f"Cloning {url}...")
    r = os.system(f"cd {WORK_DIR} && git clone {url} 2>&1")
    if r != 0:
        raise RuntimeError("git clone that bai. Thu METHOD=1 hoac METHOD=2.")

def via_kaggle_dataset(dataset_name):
    src = f"/kaggle/input/{dataset_name}"
    if not os.path.isdir(src):
        raise RuntimeError(
            f"Khong thay {src}.\n"
            "Kiem tra:\n"
            "  1. Sidebar phai -> 'Add data' -> tim 'vtai-code' -> Add\n"
            "  2. Ten dataset phai khop chinh xac (chu thuong, dau gach)"
        )
    if os.path.exists(REPO_DIR):
        shutil.rmtree(REPO_DIR)
    shutil.copytree(src, REPO_DIR)
    print(f"Copied tu {src} -> {REPO_DIR}")

def via_zip_upload(zip_basename):
    zip_path = f"/kaggle/input/{zip_basename}"
    if not os.path.exists(zip_path):
        raise RuntimeError(f"Khong thay {zip_path}")
    if os.path.exists(REPO_DIR):
        shutil.rmtree(REPO_DIR)
    os.makedirs(REPO_DIR)
    with zipfile.ZipFile(zip_path, 'r') as zf:
        zf.extractall(REPO_DIR)
    print(f"Extracted {zip_path} -> {REPO_DIR}")

# ===== FIXUP: Neu extract ra flat (KHONG co thu muc con) =====
# Kaggle cam zip co '/' trong ten file, nen phai extract flat roi sap xep lai
def fixup_flat_layout():
    """Neu KHONG co bootstrap_gt/ folder (zip bi extract flat), tao lai."""
    if os.path.exists(f"{REPO_DIR}/bootstrap_gt"):
        print("bootstrap_gt/ da ton tai -> OK")
        return
    root_jsons = sorted([f for f in os.listdir(REPO_DIR) if f.endswith(".json")])
    if not root_jsons:
        return
    print(f"Phat hien flat layout ({len(root_jsons)} JSON o root). Sap xep lai...")
    os.makedirs(f"{REPO_DIR}/bootstrap_gt", exist_ok=True)
    for fn in root_jsons:
        shutil.move(f"{REPO_DIR}/{fn}", f"{REPO_DIR}/bootstrap_gt/{fn}")
    print(f"Da chuyen {len(root_jsons)} file JSON vao bootstrap_gt/")
# ============================================================

if os.path.exists(REPO_DIR):
    print(f"Repo da ton tai: {REPO_DIR}")
else:
    if METHOD == 0:
        via_git()
    elif METHOD == 1:
        DATASET_NAME = "vtai-code"
        via_kaggle_dataset(DATASET_NAME)
    elif METHOD == 2:
        ZIP_BASENAME = "vtai-code.zip"
        via_zip_upload(ZIP_BASENAME)
    else:
        raise ValueError("METHOD phai la 0, 1, hoac 2")

fixup_flat_layout()

os.chdir(REPO_DIR)
if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)

print(f"\nWorking: {os.getcwd()}")
root_files = sorted([f for f in os.listdir('.') if not os.path.isdir(f)])
print(f"Root files ({len(root_files)}): {root_files[:15]}")
if os.path.exists('bootstrap_gt'):
    print(f"bootstrap_gt/: {len(os.listdir('bootstrap_gt'))} files")
if os.path.exists('input'):
    print(f"input/input/: {len(os.listdir('input/input'))} files")

needed = ['v20_pipeline.py', 'llm_prompts.py', 'hybrid_pipeline.py',
          'drug_dict_v3.py', 'vocab_v5.py']
missing = [f for f in needed if not os.path.exists(f)]
if missing:
    print(f"\nTHIEU file: {missing}")
else:
    print("\nTat ca file can thiet OK")

## Cell 4 - Cau hinh

DIEU CHINH CAC THAM SO O DAY THEO NHU CAU.

**Khi TRAIN = True:** cell 5 se chay training. 
**Khi TRAIN = False:** bo qua training, chay thang inference.

In [ ]:
# ===== CAU HINH ===============================
# Chi thay doi cac gia tri o day. Khong can sua cell khac.

# --- MODEL ---
BASE_MODEL = "Qwen/Qwen2.5-7B-Instruct"

# --- TRAIN ---
TRAIN = False       # True = train LoRA, False = inference thoi (da co adapter)

# LoRA params
LORA_R = 8          # rank. 8 = it VRAM nhat, van tot cho NER
LORA_ALPHA = 16     # = 2 * LORA_R
LORA_DROPOUT = 0.05
EPOCHS = 3          # 3 epoch la du voi 100 example
LR = 2e-4           # slightly higher lr cho rank thap
BATCH_SIZE = 1
GRAD_ACCUM = 4      # effective batch = 4 (giam so bat)
MAX_SEQ_LEN = 2048 # 2048 = OK voi T4 14GB + 4-bit + grad ckpt
WARMUP_RATIO = 0.1
USE_QLORA = True    # bat buoc True voi T4
GRADIENT_CKPT = True

# Adapter da train (khi TRAIN=False)
# Dataset Kaggle: bnhphongt/lora-adapter (persist qua cac session)
LORA_ADAPTER = "/kaggle/input/lora-adapter"

# --- INFERENCE ---
TEST_INPUT_DIR = "input/input"
HYBRID = True       # True = rules+v20 + LLM, False = LLM thuan
RULE_WEIGHT = 0.6

# --- OUTPUT ---
OUTPUT_DIR = "/kaggle/working/output_hybrid"
ZIP_PATH = "/kaggle/working/output_hybrid.zip"


print(f"Model      : {BASE_MODEL}")
print(f"Train      : {TRAIN}")
if TRAIN:
    print(f"  LoRA: r={LORA_R}, alpha={LORA_ALPHA}, seq={MAX_SEQ_LEN}, bs={BATCH_SIZE}x{GRAD_ACCUM}")
    print(f"  Epochs={EPOCHS}, lr={LR}, qlora={USE_QLORA}, grad_ckpt={GRADIENT_CKPT}")
else:
    print(f"  Adapter: {LORA_ADAPTER}")
print(f"Hybrid     : {HYBRID}, rule_weight={RULE_WEIGHT}")
print(f"Test input  : {TEST_INPUT_DIR}")
print(f"Output     : {OUTPUT_DIR}")

## Cell 5 - Train LoRA (chi chay khi TRAIN = True)

In [ ]:
if not TRAIN:
    print("TRAIN=False -> Bo qua training. Chay Cell 7 de load model.")
else:
    import os, json
    # Chong fragmentation - tranh OOM
    os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")

    from pathlib import Path
    import torch
    from transformers import (
        AutoModelForCausalLM, AutoTokenizer, TrainingArguments, Trainer, BitsAndBytesConfig,
    )
    from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training, TaskType
    from datasets import Dataset
    from llm_prompts import SYSTEM_PROMPT, FEW_SHOT_EXAMPLES

    ADAPTER_DIR = "/kaggle/working/lora_adapter"
    os.makedirs(ADAPTER_DIR, exist_ok=True)

    print("=" * 60)
    print("BAT DAU TRAINING")
    print(f"  Model: {BASE_MODEL}")
    print(f"  LoRA: r={LORA_R}, alpha={LORA_ALPHA}, seq={MAX_SEQ_LEN}")
    print(f"  Batch: {BATCH_SIZE}x{GRAD_ACCUM}, Epochs: {EPOCHS}, lr={LR}")
    print(f"  QLoRA: {USE_QLORA}, GradCkpt: {GRADIENT_CKPT}")
    print("=" * 60)

    # === 1. Build SFT dataset ===
    print("\n[1/5] Loading bootstrap_gt...")
    gt_path = Path("bootstrap_gt")
    in_path = Path("input/input")
    examples = []

    for gt_file in sorted(gt_path.glob("*.json"),
                          key=lambda p: int(p.stem) if p.stem.isdigit() else 999):
        sid = gt_file.stem
        in_file = in_path / f"{sid}.txt"
        if not in_file.exists():
            continue
        try:
            text = in_file.read_text(encoding="utf-8")
            gt = json.load(open(gt_file, encoding="utf-8"))
        except Exception:
            continue
        if not gt:
            continue
        valid = []
        for e in gt:
            pos = e.get("position", [0, 0])
            if len(pos) != 2:
                continue
            s, en = pos
            if s < 0 or en > len(text) or text[s:en] != e["text"]:
                idx = text.find(e["text"])
                if idx < 0:
                    continue
                s, en = idx, idx + len(e["text"])
            valid.append({
                "text": e["text"],
                "type": e["type"],
                "candidates": e.get("candidates", []),
                "assertions": e.get("assertions", []),
                "position": [s, en],
            })
        if not valid:
            continue
        examples.append({
            "id": sid,
            "input_text": text,
            "output_json": json.dumps(valid, ensure_ascii=False),
        })

    print(f"  {len(examples)} examples loaded")
    if len(examples) == 0:
        raise RuntimeError("Khong co example! Kiem tra bootstrap_gt/ va input/input/")

    # === 2. Tokenizer ===
    print("\n[2/5] Loading tokenizer...")
    tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL, trust_remote_code=True)
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token
    tokenizer.padding_side = "right"
    if not tokenizer.chat_template:
        tokenizer.chat_template = (
            "{% for message in messages %}"
            "{{'<|im_start|>' + message['role'] + '\\n' + message['content'] | trim + '<|im_end|>\\n'}}"
            "{% endfor %}"
            "{% if add_generation_prompt %}{{ '<|im_start|>assistant\\n' }}{% endif %}"
        )

    # === 3. Model ===
    print("\n[3/5] Loading base model (4-bit)...")
    torch.cuda.empty_cache()

    model_kwargs = {
        "trust_remote_code": True,
        "torch_dtype": torch.bfloat16,
        "device_map": "auto",
    }
    if USE_QLORA:
        model_kwargs["quantization_config"] = BitsAndBytesConfig(
            load_in_4bit=True,
            bnb_4bit_quant_type="nf4",
            bnb_4bit_compute_dtype=torch.bfloat16,
            bnb_4bit_use_double_quant=True,
        )
        model_kwargs["torch_dtype"] = torch.bfloat16

    model = AutoModelForCausalLM.from_pretrained(BASE_MODEL, **model_kwargs)
    model.config.use_cache = False

    if GRADIENT_CKPT:
        model.gradient_checkpointing_enable()

    if USE_QLORA:
        model = prepare_model_for_kbit_training(
            model,
            use_gradient_checkpointing=GRADIENT_CKPT,
        )

    # === 4. LoRA ===
    print("\n[4/5] Applying LoRA...")
    lora_config = LoraConfig(
        r=LORA_R,
        lora_alpha=LORA_ALPHA,
        lora_dropout=LORA_DROPOUT,
        bias="none",
        task_type=TaskType.CAUSAL_LM,
        target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                       "gate_proj", "up_proj", "down_proj"],
    )
    model = get_peft_model(model, lora_config)
    model.print_trainable_parameters()

    # === 5. Tokenize + Train ===
    print("\n[5/5] Tokenizing...")

    def _sys_text():
        s = SYSTEM_PROMPT + "\n\nVÍ DỤ MẪU:"
        for i, ex in enumerate(FEW_SHOT_EXAMPLES, 1):
            s += f"\n\nVăn bản {i}: {ex['text']}\nJSON: {ex['output']}"
        return s

    def tokenize_fn(ex):
        # Cat text de fit trong MAX_SEQ_LEN (co du cho prompt + response)
        user_text = ex["input_text"][:MAX_SEQ_LEN // 2]  # ~1024 chars
        user_msg = f"Trích xuất thực thể y tế từ:\n{user_text}\n\nTrả lời bằng JSON array:"
        messages = [
            {"role": "system", "content": _sys_text()},
            {"role": "user", "content": user_msg},
        ]
        prompt = tokenizer.apply_chat_template(
            messages, tokenize=False, add_generation_prompt=True
        )
        response = ex["output_json"]
        full = prompt + response + tokenizer.eos_token

        prompt_ids = tokenizer(prompt, add_special_tokens=False)["input_ids"]
        full_ids = tokenizer(full, add_special_tokens=False,
                             truncation=True, max_length=MAX_SEQ_LEN)["input_ids"]

        labels = [-100] * len(prompt_ids) + full_ids[len(prompt_ids):]
        labels = labels[:len(full_ids)]

        return {
            "input_ids": full_ids,
            "labels": labels,
            "attention_mask": [1] * len(full_ids),
        }

    ds = Dataset.from_list(examples)
    ds = ds.map(tokenize_fn, remove_columns=ds.column_names, num_proc=1)
    avg_len = sum(len(x) for x in ds["input_ids"]) / len(ds)
    print(f"  {len(ds)} examples, avg {avg_len:.0f} tokens, max {MAX_SEQ_LEN}")

    def collator(features):
        max_len = max(len(f["input_ids"]) for f in features)
        ids, lbls, am = [], [], []
        for f in features:
            pad = max_len - len(f["input_ids"])
            ids.append(f["input_ids"] + [tokenizer.pad_token_id] * pad)
            lbls.append(f["labels"] + [-100] * pad)
            am.append(f["attention_mask"] + [0] * pad)
        return {
            "input_ids": torch.tensor(ids, dtype=torch.long),
            "labels": torch.tensor(lbls, dtype=torch.long),
            "attention_mask": torch.tensor(am, dtype=torch.long),
        }

    targs = TrainingArguments(
        output_dir=ADAPTER_DIR,
        num_train_epochs=EPOCHS,
        per_device_train_batch_size=BATCH_SIZE,
        gradient_accumulation_steps=GRAD_ACCUM,
        learning_rate=LR,
        warmup_ratio=WARMUP_RATIO,
        logging_steps=5,
        save_strategy="epoch",
        eval_strategy="no",
        lr_scheduler_type="cosine",
        optim="paged_adamw_8bit" if USE_QLORA else "adamw_torch",
        bf16=True,
        report_to="none",
        seed=42,
        max_grad_norm=0.5,
        gradient_checkpointing=GRADIENT_CKPT,
        remove_unused_columns=False,
    )

    trainer = Trainer(
        model=model,
        args=targs,
        train_dataset=ds,
        data_collator=collator,
    )

    print("\nBat dau training (co the mat 5-10 phut)...")
    trainer.train()

    print(f"\nSaving to {ADAPTER_DIR}...")
    model.save_pretrained(ADAPTER_DIR)
    tokenizer.save_pretrained(ADAPTER_DIR)
    print("HOAN TAT TRAINING!")

## Cell 6 - Kiem tra adapter (chi khi TRAIN = True)

In [ ]:
import os

ADAPTER_DIR = "/kaggle/working/lora_adapter"
print(f"=== Checking adapter: {ADAPTER_DIR} ===")
if os.path.exists(ADAPTER_DIR):
    total = 0
    for f in sorted(os.listdir(ADAPTER_DIR)):
        path = f"{ADAPTER_DIR}/{f}"
        sz = os.path.getsize(path) / 1024 / 1024
        total += sz
        print(f"  {f}: {sz:.2f} MB")
    print(f"\n  Total: {total:.1f} MB")
else:
    print("  WARNING: Adapter not found! Run Cell 5 first if TRAIN=True.")

## Cell 7 - Load model + LoRA adapter

In [ ]:
import os, torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import PeftModel

print(f"BASE_MODEL = {BASE_MODEL}")
print(f"LORA_ADAPTER = {LORA_ADAPTER}")
print(f"Adapter exists: {os.path.exists(LORA_ADAPTER) if LORA_ADAPTER else False}")

if LORA_ADAPTER:
    print(f"\nFiles in adapter dir:")
    for f in sorted(os.listdir(LORA_ADAPTER)):
        sz = os.path.getsize(f"{LORA_ADAPTER}/{f}") / 1024 / 1024
        print(f"  {f}: {sz:.2f} MB")

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
)

print("\nLoading tokenizer...")
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "left"

print("Loading base model (4-bit)...")
model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    quantization_config=bnb_config,
    device_map="auto",
    torch_dtype=torch.bfloat16,
    trust_remote_code=True,
)
model.config.use_cache = True

adapter_to_use = LORA_ADAPTER if LORA_ADAPTER and os.path.exists(LORA_ADAPTER) else None
if adapter_to_use:
    print(f"\nLoading LoRA adapter from: {adapter_to_use}")
    model = PeftModel.from_pretrained(model, adapter_to_use)
    print("LoRA adapter loaded!")
else:
    print("\nWARNING: No LoRA adapter - using base model (zero-shot)")

model.eval()
print("\nModel ready for inference")

## Cell 8 - Chay Inference tren test set

Xu ly tat ca file .txt trong TEST_INPUT_DIR, xuat .json ra OUTPUT_DIR.

In [ ]:
import sys, os, json, time, re
from pathlib import Path
import torch

# Them working directory vao path (sau restart thi cell 3 bi mat)
REPO_DIR = "/kaggle/working/Viettel-AI-Race"
if os.path.exists(REPO_DIR) and REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)
    os.chdir(REPO_DIR)

import v20_pipeline
from hybrid_pipeline import merge_rule_and_llm
from llm_prompts import SYSTEM_PROMPT

MAX_NEW_TOKENS = 1500
DEBUG_SAMPLE = True   # True = in ra LLM raw output cho file dau tien (debug)


def generate_with_model(text):
    """Mot LLM call, tra ve raw text."""
    msgs = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": f"Trích xuất thực thể y tế từ:\n{text[:1800]}\n\nTrả lời bằng JSON array:"},
    ]
    prompt = tokenizer.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=MAX_SEQ_LEN).to(model.device)
    with torch.no_grad():
        out = model.generate(
            **inputs,
            max_new_tokens=MAX_NEW_TOKENS,
            do_sample=False,
            repetition_penalty=1.05,
        )
    return tokenizer.decode(out[0][inputs.input_ids.shape[1]:], skip_special_tokens=True)


def parse_llm_output(raw_output):
    """Parse JSON tu LLM raw output."""
    patterns = [
        r'\[\s*\{[\s\S]*?\}\s*\]',
        r'\{[\s\S]*"entities"[\s\S]*\}',
    ]
    for pat in patterns:
        m = re.search(pat, raw_output)
        if m:
            try:
                parsed = json.loads(m.group())
                if isinstance(parsed, dict) and "entities" in parsed:
                    parsed = parsed["entities"]
                if isinstance(parsed, list):
                    return parsed
            except:
                continue
    return []


def validate_and_fix(entity, text):
    """Kiem tra position, tu dong fix neu can."""
    e = dict(entity)
    if "text" not in e or "type" not in e:
        return None
    etext = e["text"]
    if "start" in e and "end" in e and text[e["start"]:e["end"]] == etext:
        return e
    idx = text.find(etext)
    if idx >= 0:
        e["start"] = idx
        e["end"] = idx + len(etext)
    return e


def clean_entities(entities, text):
    """Loai bo hallucination, fix positions, normalize types."""
    # CHINH XAC: dung Unicode, khong phai ASCII!
    VALID_TYPES = {"THUỐC", "TRIỆU_CHỨNG", "CHẨN_ĐOÁN"}
    # Map tu ASCII -> Unicode de nhan dang
    TYPE_MAP = {
        "THUOC": "THUỐC",
        "TRIEU_CHUNG": "TRIỆU_CHỨNG",
        "CHAN_DOAN": "CHẨN_ĐOÁN",
    }
    STOPWORDS = {"khong", "khong co", "neu", "nao", "nay", "benh nhan", "nam", "nu", "tuoi",
                 "bệnh", "triệu chứng", "thuốc"}
    result = []
    seen = set()
    for e in entities:
        etype = e.get("type", "")
        # Map ASCII -> Unicode
        etype = TYPE_MAP.get(etype, etype)
        if etype not in VALID_TYPES:
            continue
        etext = e.get("text", "").strip()
        if len(etext) < 2:
            continue
        if etext.lower() in STOPWORDS:
            continue
        e2 = validate_and_fix(e, text)
        if e2 is None:
            continue
        e2["type"] = etype  # ensure Unicode
        key = (e2["type"], e2["start"], e2["end"])
        if key not in seen:
            seen.add(key)
            result.append(e2)
    return result


def extract_llm(text):
    """LLM-only extraction."""
    raw = generate_with_model(text)
    entities = parse_llm_output(raw)
    return clean_entities(entities, text)


def hybrid_extract(text):
    """Hybrid: rules (v20) + LLM."""
    rule_ents = v20_pipeline.extract_entities_v20(text)
    llm_ents = extract_llm(text)
    return merge_rule_and_llm(rule_ents, llm_ents, rule_weight=RULE_WEIGHT)


extractor = hybrid_extract if HYBRID else extract_llm
mode_label = "HYBRID" if HYBRID else "LLM-only"
print(f"Mode: {mode_label} (rule_weight={RULE_WEIGHT})")

os.makedirs(OUTPUT_DIR, exist_ok=True)
test_files = sorted(Path(TEST_INPUT_DIR).glob("*.txt"),
                     key=lambda p: int(p.stem) if p.stem.isdigit() else 999)

print(f"\nProcessing {len(test_files)} files...")
t0 = time.time()
total_ents = 0
type_counts = {"THUỐC": 0, "TRIỆU_CHỨNG": 0, "CHẨN_ĐOÁN": 0}
llm_only_count = 0

for i, fpath in enumerate(test_files, 1):
    text = fpath.read_text(encoding="utf-8")
    rule_ents = v20_pipeline.extract_entities_v20(text)
    llm_ents = extract_llm(text)

    if DEBUG_SAMPLE and i == 1:
        raw = generate_with_model(text)
        print(f"\n[DEBUG] Raw LLM output (first 500 chars):\n{raw[:500]}")
        print(f"[DEBUG] rule_ents: {len(rule_ents)}, llm_ents: {len(llm_ents)}")
        DEBUG_SAMPLE = False

    ents = merge_rule_and_llm(rule_ents, llm_ents, rule_weight=RULE_WEIGHT)

    # Dem LLM-only entities (co trong LLM nhung khong co trong rules)
    rule_keys = {(e.get('type'), e.get('start'), e.get('end')) for e in rule_ents}
    llm_only = sum(1 for e in ents if
                    (e.get('type'), e.get('start'), e.get('end')) not in rule_keys)
    llm_only_count += llm_only

    with open(f"{OUTPUT_DIR}/{fpath.stem}.json", "w", encoding="utf-8") as f:
        json.dump(ents, f, ensure_ascii=False, indent=2)

    total_ents += len(ents)
    for e in ents:
        type_counts[e.get("type", "?")] = type_counts.get(e.get("type", "?"), 0) + 1

    if i % 20 == 0 or i == len(test_files):
        elapsed = time.time() - t0
        eta = elapsed / i * (len(test_files) - i)
        print(f"  [{i}/{len(test_files)}] {elapsed:.0f}s | ETA {eta:.0f}s | "
              f"ents={total_ents} | llm_only={llm_only_count}")

print(f"\nHoan tat! {len(test_files)} files")
print(f"Total entities: {total_ents} (LLM-only: {llm_only_count})")
print(f"By type: {type_counts}")
print(f"Time: {time.time()-t0:.0f}s")
print(f"Output: {OUTPUT_DIR}")

## Cell 9 - Tao file submit .zip

In [ ]:
import zipfile, os
from pathlib import Path

print(f"Creating {ZIP_PATH}...")

with zipfile.ZipFile(ZIP_PATH, "w", zipfile.ZIP_DEFLATED) as zf:
    for f in sorted(Path(OUTPUT_DIR).glob("*.json")):
        zf.write(f, f.name)

zip_size = os.path.getsize(ZIP_PATH) / 1024
n_files = len(list(Path(OUTPUT_DIR).glob("*.json")))

print(f"ZIP: {ZIP_PATH}")
print(f"Files: {n_files}, Size: {zip_size:.1f} KB")

with zipfile.ZipFile(ZIP_PATH, "r") as zf:
    names = sorted(zf.namelist())
    print(f"Verified: {len(names)} entries")
    print(f"Sample: {names[:5]}")

print("\n=== SAN SANG SUBMIT! ===")
print(f"Download: {ZIP_PATH}")